# 05f — Freeze allocator after independent transfer confirmation

This notebook is the **method-selection lock**. It does not read test images.

Preconditions:
- 05d fixed the candidate alpha before seeing the independent CNN result;
- 05e repaired and sanity-checked the independent detector;
- 05e returned `TRANSFER_CONFIRMED_PRIMARY_ENDPOINT`.

If all provenance checks pass, this notebook freezes the allocator in
`/workspace/config/frozen_allocator.json`.

After this notebook succeeds, **do not tune alpha, payload levels, risk model, strategy definitions, or inclusion rules using test results**.


In [ ]:
from pathlib import Path
import json, shutil, yaml

from rdhlab.freeze_protocol import (
    sha256_file, build_allocator_freeze, atomic_write_json
)

config_path=Path('/workspace/config/experiment.yaml')
config=yaml.safe_load(config_path.read_text())

decision_path=Path('/workspace/results/cnn_repair_transfer/decision.json')
rule_path=Path('/workspace/results/cnn_transfer/candidate_alpha_rule.json')
payload_path=Path('/workspace/config/frozen_payloads.json')
manifest_path=Path(config['dataset']['prepared_manifest'])
risk_path=Path('/workspace/results/models/srm_teacher_local_risk.joblib')
cnn_path=Path('/workspace/results/models/enhanced_residual_cnn_05e.pt')

required=[decision_path,rule_path,payload_path,manifest_path,risk_path,cnn_path]
missing=[str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing freeze prerequisites: '+repr(missing))

decision=json.loads(decision_path.read_text())
rule=json.loads(rule_path.read_text())
payload_freeze=json.loads(payload_path.read_text())

freeze=build_allocator_freeze(
    decision,rule,payload_freeze,
    decision_sha256=sha256_file(decision_path),
    rule_sha256=sha256_file(rule_path),
    payload_freeze_sha256=sha256_file(payload_path),
    dataset_manifest_sha256=sha256_file(manifest_path),
    local_risk_model_sha256=sha256_file(risk_path),
    enhanced_cnn_sha256=sha256_file(cnn_path),
)
print(json.dumps(freeze,indent=2))


In [ ]:
freeze_dir=Path('/workspace/results/freeze')
freeze_dir.mkdir(parents=True,exist_ok=True)
target=Path('/workspace/config/frozen_allocator.json')

# Preserve any pre-05f allocator file for provenance. Never use it after this freeze.
if target.exists():
    backup=freeze_dir/'frozen_allocator_pre_05f.json'
    if not backup.exists():
        shutil.copy2(target,backup)
        print('Backed up previous allocator to',backup)

atomic_write_json(target,freeze)
atomic_write_json(freeze_dir/'frozen_allocator.json',freeze)

lock_note={
    'status':'METHOD_SELECTION_LOCKED',
    'alpha':freeze['alpha'],
    'payload_levels':freeze['payload_levels'],
    'risk_model':'srm_teacher_local_risk.joblib',
    'independent_transfer_decision':freeze['independent_transfer_decision'],
    'test_images_read_by_05f':False,
    'instruction':'Proceed to 06. Do not retune from test results.',
}
atomic_write_json(freeze_dir/'METHOD_SELECTION_LOCK.json',lock_note)

assert sha256_file(target)==sha256_file(freeze_dir/'frozen_allocator.json')
print('\nFREEZE COMPLETE')
print('alpha =',freeze['alpha'])
print('weights =',freeze['weights'])
print('payloads =',freeze['payload_levels'])
print('allocator sha256 =',sha256_file(target))
print('\nNEXT: run 06_frozen_test_experiment.ipynb exactly as frozen.')


## Interpretation

A successful run establishes the final joint score

\[
J_i = 0.25\,\widetilde P_i - 0.75\,\widetilde D_i
\]

for the current experiment, with the exact value taken from the validated freeze file rather than hard-coded by the test notebook.

The test stage may now estimate performance, but it must not be used to revise the method.
